In [0]:
%sql
mvp_eng_dados.mvp_cancer.brz_clinical_trials
mvp_eng_dados.mvp_cancer.brz_population
mvp_eng_dados.mvp_cancer.slv_conditions
mvp_eng_dados.mvp_cancer.slv_interventions
mvp_eng_dados.mvp_cancer.slv_locations
mvp_eng_dados.mvp_cancer.slv_locations_iso3
mvp_eng_dados.mvp_cancer.slv_population
mvp_eng_dados.mvp_cancer.slv_studies

mvp_eng_dados.mvp_cancer.gld_dim_country
mvp_eng_dados.mvp_cancer.gld_dim_date
mvp_eng_dados.mvp_cancer.gld_dim_intervention
mvp_eng_dados.mvp_cancer.gld_dim_sponsor
mvp_eng_dados.mvp_cancer.gld_dim_study
mvp_eng_dados.mvp_cancer.gld_fat_clinical_study
mvp_eng_dados.mvp_cancer.gld_fat_country_population
mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_intervention
mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_location
mvp_eng_dados.mvp_cancer.gld_flat_country_year_metrics

In [0]:
%run ./config

In [0]:
silver_tables = ["slv_conditions",
                 "slv_interventions",
                 "slv_locations",
                 "slv_locations_iso3",
                 "slv_population",
                 "slv_studies"]

profile_results = reduce(lambda left, right: left.unionByName(right), [profile_table(table) for table in silver_tables])

In [0]:
profile_results.write.format("delta") \
                      .mode("overwrite") \
                      .option("overwriteSchema", "true")\
                      .saveAsTable(f"{CATALOG}.{SCHEMA}.sys_data_quality_control")

In [0]:
display(profile_results.orderBy(F.desc("null_percentage")))

In [0]:
quality_rules = [
    {"rule": "NCT_ID_NULL",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """SELECT COUNT(*) AS failures
                  FROM slv_studies
                 WHERE nct_id IS NULL"""},
    
    {"rule": "NCT_ID_INVALID_FORMAT",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """ SELECT COUNT(*) AS failures
                   FROM slv_studies
                  WHERE nct_id NOT RLIKE '^NCT[0-9]{8}$'"""},
    
    {"rule": "DUPLICATE_STUDY",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """SELECT COUNT(*) AS failures
                  FROM (SELECT nct_id
                          FROM slv_studies
                         GROUP BY nct_id
                        HAVING COUNT(*) > 1)"""},
    
    {"rule": "EMPTY_STUDY_TITLE",
    "table": "slv_studies",
 "severity": "WARNING",
      "sql": """SELECT COUNT(*) AS failures
                  FROM slv_studies
                 WHERE study_title IS NULL OR TRIM(study_title) = ''"""},
    
    {"rule": "NEGATIVE_ENROLLMENT",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """ SELECT COUNT(*) AS failures
                   FROM slv_studies
                  WHERE enrollment_count < 0"""
    },

    {"rule": "INVALID_DATE_INTERVAL",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """SELECT COUNT(*) AS failures
                  FROM slv_studies
                 WHERE completion_date < start_date"""},
    
    {"rule": "INVALID_STATUS",
    "table": "slv_studies",
 "severity": "ERROR",
      "sql": """ SELECT COUNT(*) AS failures
                   FROM slv_studies
                  WHERE overall_status NOT IN ('NOT_YET_RECRUITING',
                                               'RECRUITING',
                                               'ENROLLING_BY_INVITATION',
                                               'ACTIVE_NOT_RECRUITING',
                                               'SUSPENDED',
                                               'TERMINATED',
                                               'COMPLETED',
                                               'WITHDRAWN',
                                               'UNKNOWN')"""},
    
    {"rule": "INVALID_LATITUDE",
    "table": "slv_locations_iso3",
 "severity": "ERROR",
      "sql": """ SELECT COUNT(*) AS failures
                   FROM slv_locations_iso3
                  WHERE latitude IS NOT NULL
                    AND latitude NOT BETWEEN -90 AND 90 """},
    
    {"rule": "INVALID_LONGITUDE",
    "table": "slv_locations_iso3",
 "severity": "ERROR",
      "sql": """SELECT COUNT(*) AS failures
                  FROM slv_locations_iso3
                 WHERE longitude IS NOT NULL
                   AND longitude NOT BETWEEN -180 AND 180"""},
    
    {"rule": "UNMAPPED_COUNTRY",
    "table": "slv_locations_iso3",
 "severity": "WARNING",
      "sql": """SELECT COUNT(*) AS failures
                  FROM slv_locations_iso3
                 WHERE country_iso3 IS NULL"""},
    
    {"rule": "EMPTY_INTERVENTION",
    "table": "slv_interventions",
        "severity": "ERROR",
        "sql": """
            SELECT COUNT(*) AS failures
            FROM slv_interventions
            WHERE intervention_name IS NULL
               OR TRIM(intervention_name) = ''"""},
    
    {"rule": "INVALID_POPULATION",
        "table": "slv_population",
        "severity": "ERROR",
        "sql": """
            SELECT COUNT(*) AS failures
            FROM slv_population
            WHERE population IS NULL OR population < 0"""},
    
    {"rule": "DUPLICATE_POPULATION",
        "table": "slv_population",
        "severity": "ERROR",
        "sql": """SELECT COUNT(*) AS failures
                    FROM (SELECT country_iso3, year
                            FROM slv_population
                           GROUP BY country_iso3, year
                          HAVING COUNT(*) > 1)"""},
    
    {"rule": "ORPHAN_LOCATION",
        "table": "slv_locations_iso3",
        "severity": "ERROR",
        "sql": """SELECT COUNT(*) AS failures
                    FROM slv_locations_iso3 location
               LEFT JOIN silver_studies study
                      ON location.nct_id = study.nct_id
                   WHERE study.nct_id IS NULL"""},
    
    {"rule": "ORPHAN_INTERVENTION",
    "table": "slv_interventions",
 "severity": "ERROR",
      "sql": """ SELECT COUNT(*) AS failures
                   FROM slv_interventions intervention
              LEFT JOIN silver_studies study
                     ON intervention.nct_id = study.nct_id
                  WHERE study.nct_id IS NULL"""}]

In [0]:
quality_results = []

for rule in quality_rules:
    query = rule["sql"].replace("silver_", f"`{CATALOG}`.`{SCHEMA}`.`silver_")

    failures = spark.sql(query).first()["failures"]

    quality_results.append((rule["rule"],
                            rule["table"],
                            rule["severity"],
                            int(failures),
                            failures == 0))

quality_df = spark.createDataFrame(quality_results,["rule_name",
                                                    "table_name",
                                                    "severity",
                                                    "failure_count",
                                                    "passed"])

In [0]:
(quality_df.withColumn("execution_timestamp", F.current_timestamp())
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.data_quality_results"))

In [0]:
display(quality_df.orderBy("passed",F.desc("failure_count")))

In [0]:
%sql
SELECT overall_status,
       COUNT(*) AS study_count,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS percentage
  FROM silver_studies
 GROUP BY overall_status
 ORDER BY study_count DESC;

In [0]:
%sql
SELECT COALESCE(phase, 'NA') AS phase,
       COUNT(*) AS study_count
  FROM silver_studies
 GROUP BY COALESCE(phase, 'NA')
 ORDER BY study_count DESC;

In [0]:
%sql
SELECT intervention_type,
       COUNT(DISTINCT nct_id) AS study_count
  FROM silver_interventions
 GROUP BY intervention_type
 ORDER BY study_count DESC;

In [0]:
%sql
SELECT severity,
       passed,
       COUNT(*) AS rules,
       SUM(failure_count) AS failures
  FROM data_quality_results
 GROUP BY severity, passed
 ORDER BY severity, passed;